In [ ]:
import torch

from main.model.downstream.fusion_probe.datamodule import FusionDataModule

dm = FusionDataModule(seed=1, batch_size=8)

In [ ]:
dm.add_dataset("/home/jacopo/dataset/EEGAVI/FUSION-DOWNSTREAM/DOWNSTREAM/interleaved-downstream", 1,
               valid_fraction=0.1)
dm.add_dataset("/home/jacopo/dataset/EEGAVI/FUSION-DOWNSTREAM/DOWNSTREAM/interleaved-downstream-dreamer", 1,
               valid_fraction=0.1)
dm.add_dataset("/home/jacopo/dataset/EEGAVI/FUSION-DOWNSTREAM/DOWNSTREAM/interleaved-downstream-deap", 1,
               test_fraction=1.0)

dm.setup("")

In [16]:


dl = dm.train_dataloader()
it = iter(dl)

scores_value = 0
score_count = 0
for i in it:
    try:
        scores = i["assessment", "scores"][:, 3]  # shape [B, T]
        valid = ~torch.isnan(scores)
        scores_value += scores[valid].sum().item()
        score_count += valid.sum().item()
    except:
        print(i["assessment", "scores"])

baseline = scores_value / score_count
print("baseline =", baseline)

baseline = 4.484332502512564


In [19]:
all_y = []

for batch in dm.test_dataloader():
    scores = batch["assessment", "scores"][..., 3]  # shape [B, T]
    valid = ~torch.isnan(scores)
    all_y.append(scores[valid])

y = torch.cat(all_y).float()

baseline_mse = ((y - baseline) ** 2).mean()
baseline_rmse = baseline_mse.sqrt()
baseline_mae = (y - baseline).abs().mean()

print("baseline_mse =", baseline_mse / 64)
print("baseline_rmse =", baseline_rmse /8)
print("baseline_mae =", baseline_mae / 8)

baseline_mse = tensor(0.0895)
baseline_rmse = tensor(0.2991)
baseline_mae = tensor(0.2660)


In [ ]:
baseline_mse